In [1]:
from tqdm import tqdm
import feedparser
import json
import os
import feedparser
from bs4 import BeautifulSoup
from tqdm import tqdm
import html
import re

def html_to_text(s):
    if not s:
        return ""
    s = html.unescape(s)
    soup = BeautifulSoup(s, "html.parser")
    text = soup.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def collect_from_rss_feeds(rss_feeds, max_docs=200, doc_prefix="en"):
    docs = []
    seen_urls = set()
    doc_i = 0

    for rss_url in tqdm(rss_feeds, desc="Processing RSS feeds"):
        feed = feedparser.parse(rss_url)

        for entry in feed.entries:
            url = getattr(entry, "link", "").strip()
            if not url or url in seen_urls:
                continue

            title = getattr(entry, "title", "").strip()
            summary_raw = getattr(entry, "summary", "").strip()
            summary_text = html_to_text(summary_raw)

            date = getattr(entry, "published", "")

            seen_urls.add(url)

            docs.append({
                "doc_id": f"{doc_prefix}_{doc_i:06d}",
                "title": html_to_text(title),
                "body": summary_text,
                "url": url,
                "date": date,
                "language": doc_prefix,
                "token_count": len(summary_text.split())
            })

            doc_i += 1

            if len(docs) >= max_docs:
                return docs

    return docs

rss_feeds=[
    "https://www.dhakatribune.com/feed/",
]


saving_path = r"C:\Users\RAZER\Desktop\Cross-Lingual-Information-Retrieval-System-main\Cross-Lingual-Information-Retrieval-System-main\data"
file_path = os.path.join(saving_path, "document_en_2.json")

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        existing_docs = json.load(f)
else:
    existing_docs = []

existing_urls = {doc["url"] for doc in existing_docs}

new_docs = collect_from_rss_feeds(rss_feeds)
new_docs = [doc for doc in new_docs if doc["url"] not in existing_urls]

start_id = len(existing_docs)
for i, doc in enumerate(new_docs):
    doc["doc_id"] = f"en_{start_id + i:06d}"

all_docs = existing_docs + new_docs

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(all_docs, f, ensure_ascii=False, indent=2)

print(f"Added {len(new_docs)} new documents. Total: {len(all_docs)}")

Processing RSS feeds: 100%|██████████| 1/1 [00:01<00:00,  1.49s/it]


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\RAZER\\Desktop\\Cross-Lingual-Information-Retrieval-System-main\\Cross-Lingual-Information-Retrieval-System-main\\data\\document_en_2.json'